# ⚖️ Baseline vs. Improved: what happens behind the scenes

We follow one example query through both systems:

> *"uncrowded reef break on the east coast tomorrow morning"*

---

## 🔤 Baseline (BM25): pure word matching

The baseline treats the query as a list of words.

1. **Simplify the words:** lowercase, remove filler words such as *on* and *the*, and cut words down to their stem (*uncrowded* → *uncrowd*, *breaks* → *break*).
2. **Search every spot description:** how often do these words appear there?
3. **Award points (BM25 formula):**
   - **Rare words count more:** *reef* appears in only a few spots, *break* in almost all of them.
   - **Repeated words** earn more points, but with diminishing returns.
   - **Long descriptions** are scored down slightly, so they don't win just by being long.
4. **Sort by points.** Done.

### What the baseline does not know

- **Places:** *east coast* is just the word *east* plus the word *coast*, not a place. A spot in the south whose text contains *reef* and *uncrowded* can therefore win.
- **Time:** *tomorrow morning* is ignored; to the baseline these are just meaningless words.
- **Conditions:** it has no idea whether there will be any waves tomorrow. It does not use forecast data at all.

## 🧠 Improved: five extra steps

### 1. Understand the query

Fixed rules extract three things from the text:

| Detected | How | Result in the example |
|---|---|---|
| **Place** | List of places with spelling variants (*Kaohsiung = Gaoxiong = Takao*) and region phrases (*east coast*) | Regions *east* and *northeast* |
| **Skill level** | Word list (*newbie*, *first time* → beginner) | Not mentioned, so the default: *intermediate* |
| **Time** | Time rules, counted from the moment the snapshot was taken | Tomorrow 06:00–11:00 |

The system then **filters**: only east-coast spots remain. If no spot matches, the filter is relaxed and the system tells you so.

### 2. Translate surf slang

Known slang words get extra plain-English search terms, which count **half as much** as your own words. Examples: *newbie* also searches for *beginner*, *beachie* for *beach break*.

The same BM25 search as in the baseline follows, scaled to 0–1. This is the **text score**.

### 3. Score the conditions (condition fit, 0 to 1)

For every hour in the time window, the system checks six factors against the forecast, each as a value from 0 to 1:

| Factor | Question |
|---|---|
| 🌊 **Wave size** | Does the height suit the skill level? For intermediates ideally 1.0–1.5 m; above 1.8 m the score becomes 0 |
| ⏱️ **Period** | How organised are the waves? Short, wind-driven waves score lower |
| 🧭 **Swell direction** | Does the swell come from a direction that reaches this beach? |
| 💨 **Wind** | Offshore (good), onshore (bad) or calm (good)? |
| 🌙 **Tide** | Does the tide stage match what the spot likes? |
| 🏄 **Spot level** | Does the spot profile say it suits this skill level? |

The factors are **multiplied, not added**. A single knock-out factor makes the whole spot score low: perfect wind does not help when there are no waves.

For each spot, the **best hour** in the time window counts.

### 4. Combine and apply safety

- **Final score** = 0.5 × text score + 0.5 × condition score.
- For queries **without a conditions angle**, e.g. *"reef break with showers"*, the text counts more: **0.85 to 0.15**.
- **Safety rule:** if a spot is too dangerous for your level (e.g. waves too big for beginners), it drops to the **bottom** and gets a **warning**.

### 5. Explain

A sentence is built from the computed numbers. It contains only what the system actually calculated, nothing invented:

> *"Poor for intermediates. Best around Sat 07:00: 1.1 m E swell at 5 s, wind 9 km/h NNE (cross-shore), mid tide. Limited by short period."*

If a nearby buoy measured recently, that is shown too. If the measurement differs strongly from the forecast, the system corrects the next few hours.

## 👀 At a glance

| | **Baseline** | **Improved** |
|---|---|---|
| **Asks itself** | *"Which description contains these words?"* | *"Which spot suits **you**, **there** and **then**, with **these** waves?"* |
| **Uses** | Only the text of the spot profiles | Text + place list + slang list + wave and wind forecast + buoys + surf rules |
| **Same query on different days** | Always the same result | Changes with the conditions |

> 🧪 **In the evaluation** we measure how much each of these steps adds. There are variants with exactly one step switched off: without conditions (`improved_no_cf`), without query understanding (`improved_no_qu`) and without slang expansion (`improved_no_syn`).